In [1]:
from arcgis.gis import GIS
from datetime import datetime
from datetime import timezone
now = datetime.now(timezone.utc)

In [3]:
gis_premium = GIS("https://www.arcgis.com", "hubpy_test", "hubPython01")
myhub = gis_premium.hub
gis_basic = GIS("https://www.arcgis.com", "geosaurus_hub", "hubPython01")
basic_hub = gis_basic.hub
gis_portal = GIS("https://rpubs22001.ags.esri.com/portal/home/", "creator1", "portalaccount1")

ConnectionError: A connection error has occurred: HTTPSConnectionPool(host='rpubs22001.ags.esri.com', port=443): Max retries exceeded with url: /portal/sharing/rest/info?f=json (Caused by NewConnectionError('<urllib3.connection.HTTPSConnection object at 0x000002745E6EA808>: Failed to establish a new connection: [WinError 10060] A connection attempt failed because the connected party did not properly respond after a period of time, or established connection failed because connected host has failed to respond'))

### Add Site

In [3]:
#Add site
title = "Test site %s" %int(now.timestamp() * 1000)
new_site = gis_portal.sites.add(title=title)
site_id = new_site.itemid
assert new_site.title==title, new_site.title

### Search for added site

In [4]:
#Searching for site
searched = gis_portal.sites.search(title=title, owner=gis_portal.users.me.username)
assert searched[0].itemid==new_site.itemid, searched.item

### Get site

In [5]:
#Fetching site
fetched = gis_portal.sites.get(site_id)
assert fetched==searched[0], fetched

### Update site

In [6]:
#Updating site
assert new_site.tags==[]
new_site.update(site_properties={'tags': 'Hub, OpenData'})
assert 'Hub' in new_site.tags, new_site.tags

### Clone site in same org, Basic org, and Premium org

In [12]:
def collab_group_exists(initiative):
    '''
    Verifies if collab group does not exist for this initiative/site
    '''
    try:
        initiative.collab_group_id
    except:
        return 'Works as expected'
    
def follower_group_exists(initiative):
    '''
    Verifies if follower group does not exist for this initiative/site
    '''
    try:
        initiative.followers_group_id
    except:
        return 'Works as expected'

In [9]:
#Cloning in the same Portal organization
title = "Cloned site %s" %int(now.timestamp() * 1000)
cloned_site = portal_gis.sites.clone(new_site, title=title)
site_id = cloned_site.itemid
assert cloned_site.title==title, cloned_site.title
assert collab_group_exists(cloned_initiative)=='Works as expected', 'Collab group exists'
assert follower_group_exists(cloned_initiative)=='Works as expected', 'Followers group exists'

In [10]:
#Cloning in the Hub premium organization (with admin credentials)
title = "Cloned initiative %s" %int(now.timestamp() * 1000)
cloned_premium_initiative = myhub.sites.clone(new_site, title=title)
initiative_id = cloned_premium_initiative.itemid
assert cloned_premium_initiative.title==title, cloned_premium_initiative.title
assert cloned_premium_initiative.site_id
assert cloned_premium_initiative.content_group_id
assert cloned_premium_initiative.collab_group_id
assert cloned_premium_initiative.follower_group_id

In [13]:
#Cloning in the Hub Basic org (without admin credentials)
title = "Cloned initiative %s" %int(now.timestamp() * 1000)
cloned_initiative = gis_basic.sites.clone(new_site, title=title)
initiative_id = cloned_initiative.itemid
assert cloned_initiative.title==title, cloned_initiative.title
assert cloned_initiative.site_id
assert collab_group_exists(cloned_site)=='Works as expected', 'Collab group exists'
assert follower_group_exists(cloned_site)=='Works as expected', 'Followers group exists'

### Delete initiatives

In [14]:
def verify_deleted_group(gis, g_id):
    '''
    Verify group is deleted
    '''
    if gis.groups.get(g_id) is None:
        return 'Works as expected'
    

def verify_deleted_item(gis, i_id):
    '''
    Verify item is deleted
    '''
    if gis.content.get(i_id) is None:
        return 'Works as expected'

In [18]:
#Delete enterprise site
site_id = new_site.itemid
content_group_id = new_site.content_group_id
new_site.delete()
assert verify_deleted_item(gis_portal, site_id)=='Works as expected', 'site exists'
assert verify_deleted_group(gis_portal, content_group_id)=='Works as expected', 'content group exists'

In [18]:
#Delete enterprise cloned site
site_id = cloned_site.itemid
content_group_id = cloned_site.content_group_id
cloned_site.delete()
assert verify_deleted_item(gis_portal, site_id)=='Works as expected', 'site exists'
assert verify_deleted_group(gis_portal, content_group_id)=='Works as expected', 'content group exists'

In [16]:
#Delete basic cloned initiative
site_id = cloned_initiative.site_id
initiative_id = cloned_initiative.itemid
content_group_id = cloned_initiative.content_group_id
cloned_initiative.delete()
assert verify_deleted_item(gis_basic, initiative_id)=='Works as expected', 'initiative exists'
assert verify_deleted_item(gis_basic, site_id)=='Works as expected', 'site exists'
assert verify_deleted_group(gis_basic, content_group_id)=='Works as expected', 'content group exists'

In [17]:
#Delete premium cloned initiative
site_id = cloned_premium_initiative.site_id
initiative_id = cloned_premium_initiative.itemid
collab_group_id = cloned_premium_initiative.collab_group_id
content_group_id = cloned_premium_initiative.content_group_id
followers_group_id = cloned_premium_initiative.followers_group_id
cloned_premium_initiative.delete()
assert verify_deleted_item(gis_premium, initiative_id)=='Works as expected', 'initiative exists'
assert verify_deleted_item(gis_premium, site_id)=='Works as expected', 'site exists'
assert verify_deleted_group(gis_premium, collab_group_id)=='Works as expected', 'collab group exists'
assert verify_deleted_group(gis_premium, content_group_id)=='Works as expected', 'content group exists'
assert verify_deleted_group(gis_premium, followers_group_id)=='Works as expected', 'followers group exists'